# 📘 부록: RunPod GPU에서 실습하기

이 단원부터 실습은 GPU에서 돌립니다. GPU 서버는 **RunPod**에서 필요한 시간만큼 빌리고, 수업 때 쓰던 **VS Code**로 접속해 씁니다.

**이 부록의 흐름**
1. 처음 한 번: SSH 키 만들기, RunPod에 등록, VS Code 확장 설치
2. 파드(GPU 서버) 만들기
3. VS Code로 접속
4. `git clone` 으로 자료 받기, `uv sync` 로 환경 만들기
5. GPU 확인하고 교안 실행
6. 결과 내려받고 정지·종료

## 1. RunPod 기본

### 1-1. 용어

- **파드(Pod):** RunPod에서 빌리는 GPU 서버 한 대입니다. 켜져 있는 시간만큼 요금이 나갑니다.
- **템플릿(Template):** 파드에 미리 깔아 둔 프로그램 묶음입니다. 이 과정은 공식 **PyTorch** 템플릿(PyTorch·CUDA·git·uv 포함)을 씁니다.

### 1-2. 저장 공간과 요금

| 저장 공간 | 정지하면 | 종료하면 | 요금 |
|---|---|---|---|
| 컨테이너 디스크(기본 50GB) | 지워짐 | 지워짐 | 켜져 있는 동안만 |
| 네트워크 볼륨 | 남음 | 남음 | 지우기 전까지 매달 |

- **이 단원은 컨테이너 디스크만 씁니다.** 정지하면 파일이 모두 지워지므로 결과는 먼저 내려받습니다.
- **창을 닫아도 파드는 꺼지지 않습니다.** 실습이 끝나면 반드시 정지하고 종료합니다.
- 파인튜닝 단원부터는 받은 모델을 남겨 두려고 네트워크 볼륨을 씁니다.

## 2. GPU 고르기

먼저 **GPU 메모리(VRAM)** 를 봅니다. 모델과 데이터가 메모리에 다 들어가야 학습이 돌고, 모자라면 `CUDA out of memory` 오류가 납니다.

| 단계 | 권장 메모리 | GPU 예(시간당 대략) |
|---|---|---|
| 딥러닝 학습 원리·PyTorch·트랜스포머 | 16GB | RTX 2000 Ada·RTX A4000(0.25달러) |
| 파인튜닝(SFT·LoRA·QLoRA·DPO) | 24GB | RTX A5000(0.27달러)·RTX 4090(0.9달러) |
| 서빙(vLLM)·벤치마크 | 48GB | A40·RTX A6000(0.6달러) |

- **이 단원은 16GB면 충분합니다.**
- 싼 GPU는 자리가 자주 없습니다(**Out of capacity**). **Available** 탭에서 지금 빌릴 수 있는 카드를 고릅니다.
- NVIDIA GPU를 고릅니다. AMD(MI300X 등)는 CUDA를 쓰지 않아 이 과정 코드가 돌지 않습니다.

## 3. 처음 한 번만: SSH 키와 VS Code 준비

**SSH** 는 내 PC에서 원격 서버의 터미널을 여는 방법입니다. VS Code의 **Remote - SSH** 확장이 이 연결로 서버의 폴더를 내 PC 폴더처럼 열어 줍니다.

- **개인키**(`id_ed25519`): 내 PC에만 둡니다. 남에게 주지 않습니다.
- **공개키**(`id_ed25519.pub`): RunPod에 등록합니다. 짝이 맞는 개인키를 가진 PC만 접속됩니다.

이 절은 처음 한 번만 합니다.

### 3-1. SSH 키 만들기

VS Code에서 **Terminal → New Terminal** 로 내 PC 터미널을 엽니다(macOS·Windows 같은 명령).

**OpenSSH 명령(내 PC 터미널)**

```bash
# 키 한 쌍을 만듭니다. 질문 세 개는 모두 Enter
ssh-keygen -t ed25519 -C runpod
```

**셸 명령(내 PC 터미널)**

```bash
# 공개키를 띄웁니다. ssh-ed25519 로 시작하는 한 줄 전체를 복사합니다
cat ~/.ssh/id_ed25519.pub
```

`Overwrite (y/n)?` 가 나오면 이미 키가 있는 것입니다. `n` 을 누르고 그 키를 씁니다.

<img src="images/vscode_01_keygen.png" width="1050" alt="VS Code 터미널에서 ssh-keygen 을 실행한 화면. Enter로 넘긴 질문 세 줄과 cat 으로 띄운 공개키 한 줄에 빨간 사각형이 그려져 있다.">

### 3-2. RunPod에 공개키 등록

**파드를 만들기 전에** 등록합니다. 이미 만든 파드에는 들어가지 않습니다.

1. 왼쪽 메뉴 **Credentials** → **SSH Public Keys** → **Add SSH key**
2. **Title**(예: `my-laptop`)과 **Key**(복사한 공개키)를 넣고 **Add SSH key**

<img src="images/runpod_10_ssh_keys.png" width="960" alt="RunPod 콘솔 Credentials 화면. 왼쪽 메뉴 Credentials, SSH Public Keys 탭, Add SSH key 버튼에 빨간 사각형이 그려져 있다.">

<img src="images/runpod_11_add_key.png" width="560" alt="Add SSH key 창. Key 칸에 붙여 넣은 공개키와 Add SSH key 버튼에 빨간 사각형이 그려져 있다.">

### 3-3. Remote - SSH 확장 설치

VS Code 확장(Extensions)에서 `Remote - SSH` 를 검색해 Microsoft 확장을 **Install** 합니다.

<img src="images/vscode_02_remote_ssh.png" width="720" alt="VS Code 확장 검색 화면. 검색 칸의 Remote - SSH 와 Microsoft Remote - SSH 확장의 Install 버튼에 빨간 사각형이 그려져 있다.">

## 4. 파드 만들기

1. 왼쪽 메뉴 **Billing** 에서 크레딧을 충전합니다.
2. **Pods** 를 누르면 **Deploy a Pod** 화면이 열립니다. **Pod name** 을 적습니다(예: `day65-practice`).

<img src="images/runpod_01_pods.png" width="1050" alt="RunPod 콘솔의 Deploy a Pod 화면. 왼쪽 메뉴의 Pods와 화면 제목 Deploy a Pod에 빨간 사각형이 그려져 있고, 아래에 PyTorch 템플릿과 Pod name 칸이 보인다.">

3. **Set overrides** 를 눌러 세 가지가 켜져 있는지 확인하고 닫습니다. SSH 접속에 필요합니다.
   - **Start Jupyter notebook**, **SSH terminal access** 체크
   - **Exposed TCP ports** 의 `22`

<img src="images/runpod_02_template.png" width="560" alt="Edit template overrides 창. Start Jupyter notebook 과 SSH terminal access 체크 상자, Exposed TCP ports 의 22 에 빨간 사각형이 그려져 있다.">

4. **Search GPUs** 에 `RTX 2000` 을 넣고 **Available** 탭에서 카드를 고릅니다.

<img src="images/runpod_03_gpu.png" width="875" alt="Compute 칸의 GPU 목록. 검색 칸의 RTX 2000 Ada, Available 탭, 고른 GPU 카드(RTX 2000 Ada, 16GB, 시간당 0.24달러)에 빨간 사각형이 그려져 있다.">

5. 저장 공간은 기본값(Container disk 50GB, Persistent storage 비움)으로 두고, **Total / hour** 를 확인한 뒤 **Deploy Pod** 를 누릅니다.
   - 버튼이 **Deploy when available** 이면 자리가 없는 것입니다. 다른 GPU를 고릅니다.

<img src="images/runpod_04_summary.png" width="320" alt="배포 직전 오른쪽 요약 칸. Total / hour 0.25달러와 Deploy Pod 버튼에 빨간 사각형이 그려져 있다.">

## 5. VS Code로 파드에 접속하기

### 5-1. 접속 명령 복사

파드가 켜지면(1~2분) **Connect** 탭의 **SSH over exposed TCP** 명령을 복사합니다.

- 위쪽 **SSH** 칸 명령(`...@ssh.runpod.io`)은 VS Code에서 쓸 수 없습니다.

<img src="images/runpod_05_connect.png" width="598" alt="실행 중인 파드의 상세 칸. Connect 탭과 SSH over exposed TCP 제목부터 접속 명령까지에 빨간 사각형이 그려져 있다.">

### 5-2. 호스트 추가

1. 명령 팔레트(`Cmd+Shift+P`, Windows `Ctrl+Shift+P`) → **Remote-SSH: Connect to Host...**
2. **+ Add New SSH Host...**
3. 복사한 명령을 붙여 넣고 Enter
4. 설정 파일은 첫 번째(`.ssh/config`)를 고릅니다.

<img src="images/vscode_03_connect_host.png" width="640" alt="명령 팔레트에 Remote-SSH: Connect to Host 를 입력한 화면. 입력 칸과 Remote-SSH: Connect to Host... 항목에 빨간 사각형이 그려져 있다.">

<img src="images/vscode_04_add_host.png" width="640" alt="Add New SSH Host... 항목에 빨간 사각형이 그려진 화면.">

<img src="images/vscode_05_ssh_command.png" width="640" alt="Enter SSH Connection Command 입력 칸에 ssh root@주소 -p 포트 -i ~/.ssh/id_ed25519 명령을 붙여 넣은 화면.">

<img src="images/vscode_06_config_file.png" width="640" alt="Select SSH configuration file to update 목록. 첫 번째 사용자 폴더의 .ssh/config 에 빨간 사각형이 그려져 있다.">

### 5-3. 접속하고 폴더 열기

1. 다시 **Remote-SSH: Connect to Host...** 에서 파드 주소를 고릅니다. 새 창이 열립니다.
2. `Are you sure you want to continue connecting` 이 뜨면 `yes` 입력 후 Enter(글자는 점으로 보입니다).
3. 왼쪽 아래 `SSH: 주소` 가 보이면 접속된 것입니다.
4. **File → Open Folder...** 에 `/workspace/` 를 넣고 **OK**. 신뢰 여부를 물으면 **Yes, I trust the authors**.

<img src="images/vscode_07_pick_host.png" width="640" alt="Connect to Host 목록. 새로 추가된 파드 주소에 빨간 사각형이 그려져 있다.">

<img src="images/vscode_08_open_folder.png" width="640" alt="원격 창의 Open Folder 입력 칸. /workspace/ 와 OK 버튼에 빨간 사각형이 그려져 있다.">

### 5-4. 파드에 Python·Jupyter 확장 설치

확장은 파드에 따로 설치해야 합니다. 확장 화면에서 `Python` 을 검색해 **Install in SSH: 주소** 를 누르고, `Jupyter` 도 같은 방법으로 설치합니다.

<img src="images/vscode_09_remote_ext.png" width="560" alt="접속한 창의 확장 검색 화면. 검색 칸의 Python 과 Python 확장의 Install in SSH 버튼에 빨간 사각형이 그려져 있다.">

## 6. 실습 자료 받기와 파이썬 환경 만들기

접속한 창에서 **Terminal → New Terminal** 을 열면 **파드 터미널**이 열립니다. 이 절의 명령은 모두 여기서 실행합니다.

| 명령 종류 | 하는 일 | 예 |
|---|---|---|
| 리눅스 셸 명령 | 폴더 이동·목록 보기 | `cd`, `ls` |
| git 명령 | 저장소 받기 | `git clone` |
| uv 명령 | 파이썬 환경 만들기·실행 | `uv sync`, `uv run` |
| apt 명령 | 리눅스 프로그램 설치 | `apt-get install` |

### 6-1. 자료 받기

**리눅스 셸 명령(파드 터미널)**

```bash
# 작업 폴더로 이동
cd /workspace
```

**git 명령(파드 터미널)**

```bash
# 실습 저장소 받기
git clone https://github.com/paircodingofficial-cloud/enkoa-practice-deep-learning.git
```

<img src="images/vscode_10_clone.png" width="1050" alt="파드 터미널에서 git clone 을 실행한 화면. git clone 명령과 ls 로 본 저장소 파일 목록, 왼쪽 탐색기의 enkoa-practice-deep-learning 폴더에 빨간 사각형이 그려져 있다.">

### 6-2. 환경 만들기

**리눅스 셸 명령(파드 터미널)**

```bash
# 저장소 폴더로 이동
cd enkoa-practice-deep-learning
```

**uv 명령(파드 터미널)**

```bash
# pyproject.toml·uv.lock 대로 .venv 에 패키지 설치(약 1분)
uv sync
```

```bash
# 설치한 PyTorch 버전과 GPU 사용 여부 확인
uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available())"
```

`2.11.0+cu128 True` 가 나오면 준비가 끝난 것입니다.

<img src="images/vscode_11_uv_sync.png" width="1050" alt="uv sync 가 끝난 터미널. 설치된 torch==2.11.0+cu128 줄, uv run 으로 찍은 2.11.0+cu128 True, 탐색기의 .venv 폴더에 빨간 사각형이 그려져 있다.">

### 6-3. 한글 글꼴 설치

**apt 명령(파드 터미널)**

```bash
# 그래프 한글이 네모로 나오지 않게 나눔고딕 설치
apt-get update && apt-get install -y fonts-nanum
```

### 6-4. 저장소 폴더 열기

**File → Open Folder...** 에 `/workspace/enkoa-practice-deep-learning/` 을 넣고 엽니다. 저장소 폴더를 열어야 커널 목록에 `.venv` 가 나옵니다.

<img src="images/vscode_12_open_repo.png" width="640" alt="Open Folder 입력 칸에 /workspace/enkoa-practice-deep-learning/ 을 넣은 화면. 입력 칸과 OK 버튼에 빨간 사각형이 그려져 있다.">

## 7. GPU 확인하고 교안 실행하기

### 7-1. 노트북 만들고 커널 고르기

1. 명령 팔레트 → **Create: New Jupyter Notebook**, `내작업/GPU_확인.ipynb` 로 저장
2. 오른쪽 위 **Select Kernel** → **Python Environments...**
3. **.venv (Python 3.12.3)** 선택(`Recommended`)

다른 파이썬(3.10, `/usr/bin/python3` 등)에는 이 저장소의 패키지가 없습니다. 꼭 `.venv` 를 고릅니다.

<img src="images/vscode_13_select_kernel.png" width="1050" alt="노트북 오른쪽 위 Select Kernel 버튼과 커널 종류 목록의 Python Environments... 에 빨간 사각형이 그려져 있다.">

<img src="images/vscode_14_venv_kernel.png" width="640" alt="Select a Python Environment 목록. Recommended 표시가 붙은 .venv (Python 3.12.3) 에 빨간 사각형이 그려져 있다.">

### 7-2. GPU 확인

아래 두 셀을 실행합니다. `!` 는 셀에서 터미널 명령을 실행하라는 표시입니다.

```python
# GPU 이름과 메모리 확인
!nvidia-smi
```

```python
import torch

# PyTorch가 GPU를 쓸 수 있는지 확인합니다
print("PyTorch 버전:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print("GPU 이름:", torch.cuda.get_device_name(0), "| GPU 메모리(GB):", round(total_gb, 1))
```

- `GPU 사용 가능: True` 이고 버전이 `2.11.0+cu128` 이면 성공입니다.
- 버전이 `2.8` 로 시작하면 커널을 잘못 고른 것입니다. 7-1대로 `.venv` 로 바꿉니다.

<img src="images/vscode_15_gpu_check.png" width="1050" alt="VS Code에서 nvidia-smi 와 torch.cuda 확인 셀을 실행한 화면. 커널 .venv (Python 3.12.3), GPU 이름 NVIDIA RTX A6000, 메모리 49140MiB, 출력 GPU 사용 가능: True 에 빨간 사각형이 그려져 있다.">

### 7-3. 교안 실행

`day65_딥러닝_학습원리/교안_01_손실함수와_경사하강.ipynb` 를 `.venv` 커널로 위에서부터 실행합니다. 준비 셀에 `계산 장치: cuda` 가 나오면 GPU로 계산하는 것입니다(화면은 RTX A6000 파드).

<img src="images/vscode_16_lesson_cuda.png" width="1050" alt="VS Code에서 교안 01 준비 셀과 데이터 읽기 셀을 실행한 화면. 커널 .venv (Python 3.12.3), 출력 계산 장치: cuda | NVIDIA RTX A6000, 행 수: 100 에 빨간 사각형이 그려져 있다.">

## 8. 끝내기: 내려받고 정지·종료

1. **내려받기:** 탐색기에서 폴더를 오른쪽 클릭 → **Download...** → 저장 위치 선택

<img src="images/vscode_17_download.png" width="740" alt="VS Code 탐색기에서 단원 폴더를 오른쪽 클릭한 메뉴. 폴더와 Download... 항목에 빨간 사각형이 그려져 있다.">

2. **정지:** 파드 상세 칸 **Stop** → **Stop Pod**. 볼륨이 없으면 `ALL DATA will be lost!` 가 뜹니다.

<img src="images/runpod_08_stop.png" width="560" alt="정지 확인 창. 볼륨이 없어 모든 데이터가 지워진다는 경고와 Stop Pod 버튼에 빨간 사각형이 그려져 있다.">

3. **종료:** 정지 뒤 나타나는 **Terminate** → **Terminate Pod**. 마지막으로 Pods 화면이 비었는지 확인합니다.

<img src="images/runpod_09_terminate.png" width="600" alt="정지된 파드의 상세 칸. 정지한 뒤 나타난 Terminate 버튼에 빨간 사각형이 그려져 있다.">

**다음에 새 파드를 만들면:** 주소가 바뀌므로 5절부터 다시 합니다. 3절(키·확장)은 다시 하지 않습니다.

## 9. VS Code 접속이 안 될 때: Jupyter Lab

1. **Connect** 탭의 **Jupyter Lab** 링크를 엽니다(**Ready** 일 때).
2. **File → New → Terminal** 에서 6-1과 6-3을 실행합니다.
3. 아래 명령을 실행하고, 교안을 기본 커널 **Python 3 (ipykernel)** 로 엽니다.

**pip 명령(Jupyter Lab 터미널)**

```bash
# 템플릿 파이썬에 없는 두 라이브러리만 설치(PyTorch 는 이미 있음)
pip install pandas matplotlib
```

<img src="images/runpod_05b_jupyter.png" width="598" alt="파드 상세 칸의 Connect 탭. HTTP services 의 Jupyter Lab 링크와 Ready 표시에 빨간 사각형이 그려져 있다.">

<img src="images/runpod_07_lesson_cuda.png" width="760" alt="Jupyter Lab에서 교안 01 준비 셀을 실행한 화면. 출력 계산 장치: cuda 와 GPU 이름(NVIDIA A40)에 빨간 사각형이 그려져 있다.">

## 10. 자주 막히는 곳

| 증상 | 해결 |
|---|---|
| 비밀번호를 묻는다 | 공개키가 등록되지 않았습니다. 3-2를 하고 파드를 새로 만듭니다 |
| `Connection timed out` | 1분 뒤 다시 접속합니다. **SSH over exposed TCP** 가 없으면 4절 3번 설정을 켜고 새로 만듭니다 |
| `REMOTE HOST IDENTIFICATION HAS CHANGED` | 내 PC 터미널에서 `ssh-keygen -R "[주소]:포트"` 실행 후 다시 접속 |
| 커널 목록에 `.venv` 가 없다 | 6-4대로 저장소 폴더를 엽니다 |
| `No module named 'pandas'` | 커널이 `.venv` 가 아닙니다. 7-1대로 바꿉니다 |
| 준비 셀에 `cpu` | GPU 없이 만든 파드입니다. 새로 만듭니다 |
| `CUDA out of memory` | 배치를 줄이거나 메모리가 큰 GPU로 새로 만듭니다 |
| 그래프 한글이 네모 | 6-3 글꼴 설치 후 노트북 **Restart** |
| `git` 이나 `uv` 가 없다 | 아래 명령으로 설치합니다(PyTorch 템플릿에는 이미 있음) |

**apt 명령(파드 터미널)**

```bash
# git 과 curl 설치
apt-get update && apt-get install -y git curl
```

**리눅스 셸 명령(파드 터미널)**

```bash
# curl 로 uv 설치 스크립트를 받아 실행하고, 지금 터미널에서 uv 를 쓰도록 경로를 불러옵니다
curl -LsSf https://astral.sh/uv/install.sh | sh
source $HOME/.local/bin/env
```

## 11. 핵심 정리

- **처음 한 번:** SSH 키 만들기 → 공개키 등록 → Remote - SSH 설치
- **파드마다:** 접속 → 확장 설치 → `git clone` → `uv sync` → `.venv` 커널
- **끝나면:** 내려받기 → 정지 → 종료